# Linear vs cosine noise schedule on MNIST

Trains the class-conditional U-Net again with a **cosine** noise schedule (everything else unchanged) and compares it with the existing **linear** model (`checkpoints/cond_ema.pt`) using the project's classifier.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Time on a T4: about 30 min training + about 10 min evaluation. The cosine checkpoint is written to Google Drive after every epoch, so a runtime reset does not lose it.

## 1. GPU and Google Drive

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE = '/content/drive/MyDrive/diffusion-mnist-cosine'
os.makedirs(SAVE, exist_ok=True)
print('Saving to', SAVE)

## 2. Get the code

The linear model and the classifier come with the repo.

In [ ]:
%cd /content
!rm -rf diffusion-mnist
!git clone -q https://github.com/AaravKA05/diffusion-mnist
%cd /content/diffusion-mnist
!git log --oneline -1
!ls -la checkpoints
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## 3. Keep the cosine files on Drive

These links make `train.py` and `compare_schedules.py` write straight to Drive. The linear files are not touched.

In [ ]:
links = {
    'checkpoints/cosine_cond_ema.pt': 'cosine_cond_ema.pt',
    'checkpoints/cosine_cond_ckpt.pt': 'cosine_cond_ckpt.pt',
    'results/cosine_cond_loss_history.json': 'cosine_cond_loss_history.json',
    'checkpoints/cosine_ema.pt': 'cosine_ema.pt',      # optional part 7
    'checkpoints/cosine_ckpt.pt': 'cosine_ckpt.pt',
    'results/cosine_loss_history.json': 'cosine_loss_history.json',
}
for local, name in links.items():
    if os.path.lexists(local):
        os.remove(local)
    os.symlink(f'{SAVE}/{name}', local)

import json
hist = f'{SAVE}/cosine_cond_loss_history.json'
done = len(json.load(open(hist))) if os.path.exists(hist) else 0
print(f'Cosine conditional model on Drive: {done}/30 epochs finished')
if done >= 30:
    print('Training already finished: skip step 4.')

## 4. Train the cosine model (about 30 min)

Same settings as the linear model (`config.py`: 32x32, `T = 400`, 30 epochs, EMA 0.995). Only the schedule changes.

If the runtime resets during training, run steps 1–3 again and then this cell: training starts again from epoch 0 (the script has no resume).

In [ ]:
!DIFFUSION_SCHEDULE=cosine python -u train.py --cond --epochs 30 2>&1 | grep --line-buffered -E "schedule|epoch [0-9]+ +loss"

## 5. Evaluate both models (about 10 min)

For each model: DDIM 10, 20 and 50 steps × guidance `w` 1.0 and 1.2 × seeds 1, 2, 3, with 100 samples per digit. Both models start from exactly the same noise, so the comparison is paired.

In [ ]:
!python compare_schedules.py eval --schedule linear
!python compare_schedules.py eval --schedule cosine
!cp results/schedule_eval_*.json results/schedule_samples_*.png {SAVE}/ 2>/dev/null; echo saved

## 6. Results

In [ ]:
!python compare_schedules.py report
!cp results/schedule_compare.* {SAVE}/

from IPython.display import Image, display
display(Image('results/schedule_compare.png'))
print('Linear samples (DDIM 20, w = 1.2): one row per digit')
display(Image('results/schedule_samples_linear.png'))
print('Cosine samples (DDIM 20, w = 1.2)')
display(Image('results/schedule_samples_cosine.png'))

### Download

This zip has every result file. Upload it in the chat so the results can go into the README.

In [ ]:
!cd results && zip -q ../schedule_results.zip schedule_* cosine_cond_loss_history.json cosine_cond_loss_curve.png 2>/dev/null; ls -la ../schedule_results.zip
from google.colab import files
files.download('schedule_results.zip')

## 7. Optional: the digit-1 question (about 30 more min)

The README reports that the unconditional linear model makes very few 1s (14 of 500 samples). This trains an **unconditional** cosine model and counts the digits in 500 of its samples. The linear count comes from the earlier `diagnose.py` run, because the unconditional linear checkpoint is not in the repo.

In [ ]:
!DIFFUSION_SCHEDULE=cosine python -u train.py --epochs 30 2>&1 | grep --line-buffered -E "schedule|epoch [0-9]+ +loss"
!python compare_schedules.py uncond --schedule cosine
!cp results/schedule_uncond_cosine.json {SAVE}/
!python compare_schedules.py report | tail -3

Copy the printed **class counts** line into the chat (or upload `schedule_uncond_cosine.json` from the Drive folder).